In [10]:
# imports
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from IPython.display import clear_output
from openai import OpenAI


In [ ]:
# Load env variables and verify keys

load_dotenv(override=True)
openai = OpenAI(
    api_key=os.getenv("ANTHROPIC_API_KEY"),
    base_url="https://api.anthropic.com/v1/"
)

# Check the key

if not openai.api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not openai.api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif openai.api_key.strip() != openai.api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

In [12]:

def ask_ai(system, user, model="claude-haiku-4-5", stream=True):
    messages = [
        {"role": "system", "content": system},
        {"role": "user", "content": user}
    ]
    if not stream:
        response = openai.chat.completions.create(model=model, messages=messages)
        return response.choices[0].message.content

    chunks = openai.chat.completions.create(model=model, messages=messages, stream=True)
    response = ""
    for chunk in chunks:
        response += chunk.choices[0].delta.content or ""
        clear_output(wait=True)
        display(Markdown(response))
    return response


In [ ]:
system_prompt = "You're an assistant that gives concise, practical answers."
user_prompt = "Recommend 3 courses for mastering AI Engineering from scratch, with why."

for model in ["claude-haiku-4-5", "claude-sonnet-4-6"]:
    print(f"\n{'='*40}\n{model}\n{'='*40}")
    print(ask_ai(system_prompt, user_prompt, model=model, stream=False))

In [ ]:
def chat_demo(turns):
    messages = [{"role": "system", "content": "You are a helpful, concise assistant."}]
    for user_input in turns:
        print(f"You: {user_input}")
        messages.append({"role": "user", "content": user_input})
        reply = openai.chat.completions.create(
            model="claude-haiku-4-5", messages=messages
        ).choices[0].message.content
        messages.append({"role": "assistant", "content": reply})
        print(f"Assistant: {reply}\n")

chat_demo([
    "My name is Sanjit and I'm a Unity dev.",
    "What's a good first AI project for someone with my background?",
    "What was my name again?"   # tests whether it remembered
])

In [7]:
import scraper
print(dir(scraper))

['__author__', '__builtins__', '__cached__', '__doc__', '__email__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__']


In [ ]:
!pip install beautifulsoup4


In [ ]:
import requests
from bs4 import BeautifulSoup

def get_website_text(url):
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    # strip out script/style/nav noise
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    return soup.get_text(separator="\n", strip=True)

# use it
url = "https://edwarddonner.com"
site_text = get_website_text(url)
print(site_text[:500])   # preview



In [14]:
ask_ai(system_prompt, f"Here's the website content:\n\n{site_text}")

```json
[
  {
    "name": "AI Coder: Vibe Coder to Agentic Engineer",
    "price": "Not specified",
    "url": "https://www.edwarddonner.com",
    "why": "Learn LLM application development and AI engineering from Edward Donner, CTO of Nebula.io with 600,000+ students across 194 countries in his best-selling courses"
  },
  {
    "name": "AI Builder with n8n – Create Agents and Voice Agents",
    "price": "Not specified",
    "url": "https://www.edwarddonner.com",
    "why": "Master AI agent building and voice agent creation using n8n from an expert instructor with proven top-rated courses"
  },
  {
    "name": "AI Engineering MLOps Track – Deploy AI to Production",
    "price": "Not specified",
    "url": "https://www.edwarddonner.com",
    "why": "Comprehensive MLOps training for productionizing AI systems from Edward Donner's curriculum of best-selling courses"
  }
]
```

'```json\n[\n  {\n    "name": "AI Coder: Vibe Coder to Agentic Engineer",\n    "price": "Not specified",\n    "url": "https://www.edwarddonner.com",\n    "why": "Learn LLM application development and AI engineering from Edward Donner, CTO of Nebula.io with 600,000+ students across 194 countries in his best-selling courses"\n  },\n  {\n    "name": "AI Builder with n8n – Create Agents and Voice Agents",\n    "price": "Not specified",\n    "url": "https://www.edwarddonner.com",\n    "why": "Master AI agent building and voice agent creation using n8n from an expert instructor with proven top-rated courses"\n  },\n  {\n    "name": "AI Engineering MLOps Track – Deploy AI to Production",\n    "price": "Not specified",\n    "url": "https://www.edwarddonner.com",\n    "why": "Comprehensive MLOps training for productionizing AI systems from Edward Donner\'s curriculum of best-selling courses"\n  }\n]\n```'

In [ ]:
import json

system_prompt = """
You are an AI course advisor. Respond ONLY with valid JSON, no preamble,
no markdown fences. Schema:
[{"name": str, "price": str, "url": str, "why": str}]
"""
user_prompt = "Recommend 3 beginner AI Engineering courses that get you job-ready."

raw = ask_ai(system_prompt, user_prompt, stream=False)
raw = raw.strip().removeprefix("```json").removesuffix("```").strip()
courses = json.loads(raw)

table = "| Course | Price | Why |\n|---|---|---|\n"
for c in courses:
    table += f"| [{c['name']}]({c['url']}) | {c['price']} | {c['why']} |\n"

display(Markdown(table))